# Regridding

:::{warning}
This notebook is a work in progress.  The `xhycom.regrid()` method described
below is **not yet implemented**.  This document outlines the planned interface
and demonstrates the manual steps in the meantime.
:::

HYCOM output lives on two non-standard grids:

- **Horizontal** — a curvilinear grid where `lon` and `lat` are 2-D arrays,
  not 1-D axes.  Standard tools that assume a regular lat/lon grid will not
  work directly.
- **Vertical** — a hybrid isopycnal coordinate where layer thicknesses are
  stored in Pa (not metres), and the number and position of layers varies in
  space and time.

For many downstream applications (comparison with observations, forcing other
models, publication figures) you need data on a regular **lat / lon / depth**
grid.  This notebook documents the two regridding steps and the planned
`xhycom.regrid()` API that will eventually automate them.

## Planned API

The goal is a single convenience function that handles both steps:

```python
import numpy as np
import xhycom

ds = xhycom.open_mfdataset(DATA_PATH + "archm.*", grid=GRID_PATH,
                            chunks={"time": 1})

ds_reg = xhycom.regrid(
    ds,
    lon=np.arange(-30, 30.25, 0.25),   # target longitudes
    lat=np.arange(50, 80.25, 0.25),    # target latitudes
    z=np.arange(0, 510, 10),           # target depth levels in metres
    method="bilinear",                  # horizontal interpolation method
    z_method="linear",                  # vertical interpolation method
)
# ds_reg has dims (time, z, lat, lon) on a regular grid
```

The function will:
1. Compute layer interface depths from `thknss` (Pa → metres).
2. Regrid horizontally using [xESMF](https://xesmf.readthedocs.io).
3. Interpolate vertically onto the requested depth levels.
4. Propagate CF attributes and coordinates throughout.

## Setup

In [ ]:
import numpy as np
import xarray as xr
import xhycom

GRID_PATH = "/cluster/home/nlo043/NERSC-HYCOM-CICE/TP2a0.10/topo/regional.grid"
DATA_PATH = "/nird/datalake/NS9481K/shuang/TP2_output/expt_02.8/"

ds = xhycom.open_mfdataset(DATA_PATH + "archm.2020*", grid=GRID_PATH,
                            chunks={"time": 1})

## Step 1 — Horizontal regridding (curvilinear → regular lat/lon)

[xESMF](https://xesmf.readthedocs.io) is the standard tool for regridding
xarray Datasets between structured grids.  It supports bilinear, conservative,
and patch interpolation methods and produces a weight file that can be reused
across time steps.

```bash
pip install xesmf
```

In [ ]:
import xesmf as xe

# Define the target regular lat/lon grid
ds_out = xr.Dataset({
    "lat": (["lat"], np.arange(50, 80.25, 0.25)),
    "lon": (["lon"], np.arange(-30, 30.25, 0.25)),
})

# xESMF expects the source grid's lon/lat to be named 'lon' and 'lat'.
# ds already has these as non-dimension coordinates — rename for xESMF.
ds_src = ds.rename({"lon": "lon", "lat": "lat"})  # already correct names

# Build the regridder (weight file written once; reused for all time steps)
regridder = xe.Regridder(ds_src, ds_out, method="bilinear",
                          periodic=False, ignore_degenerate=True)
regridder

In [ ]:
# Apply to the full Dataset — xESMF maps over all non-spatial dimensions
ds_hor = regridder(ds_src)
ds_hor

:::{note}
U-point and V-point variables (`u-vel.`, `v_btrop`, …) live on a staggered
grid and carry `lon_u`/`lat_u` or `lon_v`/`lat_v` coordinates.  They need
a separate regridder built from those staggered coordinates.  The planned
`xhycom.regrid()` will handle this automatically.
:::

## Step 2 — Vertical regridding (hybrid isopycnal → z-levels)

HYCOM stores layer thickness as pressure (`thknss`, in Pa).  To get physical
depths we need two conversions:

1. **Pa → metres** using the hydrostatic approximation:  
   `dz [m] = thknss [Pa] / (ρ₀ × g)  ≈  thknss / 9806`

2. **Layer centres to interface depths** by cumulative sum, then
   interpolation onto requested z-levels.

In [ ]:
# 1. Layer thickness in metres
thknss_m = ds_hor["thknss"] / 9806.0          # Pa → m

# 2. Interface depths (cumulative sum from surface)
#    interface 0 = sea surface (depth = 0)
#    interface k = bottom of layer k
z_interfaces = thknss_m.cumsum("k")           # (time, k, lat, lon)

# 3. Layer-centre depths (mid-point of each layer)
z_centres = z_interfaces - thknss_m / 2       # (time, k, lat, lon)
z_centres.attrs = {"long_name": "layer centre depth", "units": "m", "positive": "down"}
z_centres

In [ ]:
# 4. Interpolate temperature onto target z-levels
#    xarray.DataArray.interp() works along a named dimension,
#    but here z_centres varies in space and time — we need a loop
#    or a dedicated tool such as xgcm.

# Simple single-column example (slow for full 3-D fields):
target_z = np.arange(0, 510, 10)   # 0–500 m in 10-m steps

# For a single lat/lon point:
temp_col = ds_hor["temp"].isel(lat=20, lon=40).compute()
z_col    = z_centres.isel(lat=20, lon=40).compute()

# Interpolate for one time step
from scipy.interpolate import interp1d
t0 = 0
f = interp1d(z_col.isel(time=t0).values, temp_col.isel(time=t0).values,
             bounds_error=False, fill_value=np.nan)
temp_z = f(target_z)

import matplotlib.pyplot as plt
plt.plot(temp_z, -target_z)
plt.xlabel('Temperature (degC)')
plt.ylabel('Depth (m)')
plt.title('Interpolated temperature profile');

### Full 3-D vertical regridding with xgcm

For the full `(time, lat, lon)` field, [xgcm](https://xgcm.readthedocs.io)
provides a vectorised vertical interpolation that works with Dask arrays:

```python
import xgcm

# Build an xgcm Grid that knows about the vertical coordinate
grid = xgcm.Grid(ds_hor, coords={"Z": {"center": "k"}}, periodic=False)

# Interpolate temp onto target z-levels using z_centres as the coordinate
temp_on_z = grid.transform(
    ds_hor["temp"],
    "Z",
    target_z,
    target_data=z_centres,
    method="linear",
)
# temp_on_z has dims (time, z, lat, lon)
```

This is the approach that `xhycom.regrid()` will wrap internally.

## Implementation roadmap for `xhycom.regrid()`

The planned implementation will:

- Accept a pre-opened `xr.Dataset` from `open_dataset` / `open_mfdataset`.
- Detect which variables are T-point, U-point, or V-point and use the correct
  source coordinates for each.
- Build xESMF weight files once and cache them for the lifetime of the call.
- Use `xgcm.Grid.transform` for vectorised vertical interpolation.
- Expose `method` (bilinear / conservative / patch) and `z_method` (linear /
  log-linear) as arguments.
- Preserve CF attributes and attach `depth` as a proper dimension coordinate
  with `positive: down`.
- Be fully lazy — return a Dask-backed Dataset that only reads and interpolates
  data when `.compute()` is called.

Contributions welcome — see the
[GitHub issues](https://github.com/nansencenter/xhycom/issues).